# 09 (v2) - Detector ONNX gap: box channels versus score channels (diagnosis only)

**Status of criterion A9b:** not met. Recorded values: 0.0018 (acceptance check), 0.0020 (re-check), 0.00198-0.00201 across six export/runtime settings on 31 October (notebook 09, `report/evidence/M8_onnx_parity_recheck_variants.json`). The threshold (max abs diff <= 0.001 over the whole raw head) is **not** changed here, and the six settings are **not** repeated.

**New hypothesis (stated before running):** the gap is concentrated in the 4 box-coordinate channels, which are in pixels (0-640), while the class-score channels (0-1) differ far less. The notebook also compares float32 PyTorch with a float64 PyTorch reference per channel group.

**What a result can and cannot show.** If supported, the report may add a separate diagnostic line (for example "score channels differ by <1e-6; box channels by about 0.002 px"); A9b still stays *not met* and the service keeps PyTorch. A code check of this notebook's tool on the public `yolo11n.pt` showed the same pattern (box 0.0025, score 1.8e-7); that is a code check only and is not evidence about the team detector.

Cost: CPU runtime, about 3-5 minutes, free. No training.


In [ ]:
# 1. Setup (pinned versions as in notebook 09)
!git clone -q --branch phase2-submission-hardening https://github.com/Yasser-20-alhajlah/smart-pet-vision /content/spv-repo 2>/dev/null || git -C /content/spv-repo pull -q
!pip -q install ultralytics==8.4.182 onnx==1.23.2 onnxruntime==1.31.0
import sys, json, hashlib, time
from pathlib import Path
sys.path[:0] = ["/content/spv-repo/smart-pet-vision/phase2/src", "/content/spv-repo/smart-pet-vision/phase2/tools"]
from google.colab import drive
drive.mount("/content/drive")
WEIGHTS = Path("/content/drive/MyDrive/capstone /smart-pet-vision/runs/detect/yolo11s_det/weights/best.pt")   # edit if moved
OUT_DIR = Path("/content/drive/MyDrive/capstone /Submission/FINAL PHASE_2/03_Evidence")
EXPECTED = "f48e4d8ea5c925c18a4b43744c2f697bfc01a0a473492275f000848125da1cd4"
if not WEIGHTS.is_file(): raise FileNotFoundError(f"Detector weights not found: {WEIGHTS}")
got = hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()
if got != EXPECTED: raise RuntimeError(f"Detector weights differ from the evaluated file ({got[:12]}...). Stopping.")
print("Weights verified:", got)


In [ ]:
# 2. Export once with the original settings (opset 17, no simplify) to a temporary copy; the original file is never touched
import shutil, tempfile
from ultralytics import YOLO
tmp = Path(tempfile.mkdtemp()); src = tmp / "best.pt"; shutil.copy2(WEIGHTS, src)
ONNX = Path(YOLO(str(src)).export(format="onnx", imgsz=640, opset=17, simplify=False, dynamic=False))
print(ONNX)


In [ ]:
# 3. Diagnosis: same protocol as A9b (seed 0, 4 random inputs, CPU), per channel group, with a float64 reference
import onnx_channel_diagnosis as diag
rec = diag.run(src, ONNX, imgsz=640, n=4, seed=0)
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
dest = OUT_DIR / f"M8_onnx_channel_diagnosis_{stamp}.json"
assert not dest.exists()
dest.write_text(json.dumps(rec, indent=2))
print(json.dumps({k: rec[k] for k in ("A9b_value_this_run", "A9b_met", "hypothesis_supported", "summary")}, indent=2))
print("Saved", dest)
print("A9b remains", "met" if rec["A9b_met"] else "NOT met", "- the criterion is unchanged by this diagnosis.")
